# Callbacks

Er zijn heel wat extra functionaliteiten die je kan toevoegen aan het trainingsproces.
De meestgebruikte zijn: 
* EarlyStopping
* ModelCheckpoint
* Tensorboard
* LearningRate schedulers

Ook kan je custom callback functions maken om je eigen functionaliteiten toe te voegen.

In de klassieke pytorch manier schrijf je je eigen trainings-lus en is het dus eenvoudig om alle gewenste code toe te voegen. 
De efficientere manier om modellen te bouwen via Keras abstraheerd echter de trainingslus en train je door middel van de fit()-methode.
Deze manier maakt het echter onmogelijk om extra functionaliteiten toe te voegen. 

Hieronder vind je een voorbeeld van hoe je callbacks kan toevoegen aan een model gebouwd met keras op een pytorch backend

In [3]:
import numpy as np
from keras_core.models import Sequential
from keras_core.layers import Dense
from keras_core.callbacks import EarlyStopping

# test data
X_train = np.random.rand(100,10)
y_train = np.random.rand(100, 1)

# maken van het model
model = Sequential([
    Dense(10, activation='relu', input_shape=(10,1)), # relu activatiefunctie want hidden layer
    Dense(1) # lineaire activatie functie want regressie
])

model.compile(optimizer='adam', loss='mean_squared_error')

# early stopping
early_stop = EarlyStopping(patience=1, monitor='val_loss')

model.fit(X_train, y_train, epochs=100, validation_split=0.2, callbacks=[early_stop])


Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.3537 - val_loss: 0.3002
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3580 - val_loss: 0.2893
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3588 - val_loss: 0.2788
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3286 - val_loss: 0.2686
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2949 - val_loss: 0.2588
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3125 - val_loss: 0.2492
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2881 - val_loss: 0.2399
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2864 - val_loss: 0.2311
Epoch 9/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2684 - val_loss: 0.2226
Epoch 10/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2629 - val_loss: 0.2145
Epoch 11/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2472 - val_loss: 0.2069
Epoch 12/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2570 - val_loss: 0.1995
E

# Dashboard for opvolgen experimenten, evaluaties, ....

Hieronder staat een voorbeelden van hoe een dashboard kan gebruikt worden met pytorch.
Ik heb hieronder gekozen voor mlflow maar alternatieven zijn tensorboard, wandb, visdom, ...

In [ ]:
!pip install mlflow

In [8]:
import mlflow
import mlflow.pytorch

import torch
import torch.nn as nn
import torch.optim as optim

# dummy data
X = torch.randn(100,10)
y = torch.randn(100,1)

# model
model = nn.Sequential(
    nn.Linear(10,10),
    nn.ReLU(),
    nn.Linear(10,1)
)

# loss en optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# trainen
with mlflow.start_run(run_name='pytorch_example'):
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        # logging per epoch
        mlflow.log_metric('loss', loss.item(), step=epoch)

    mlflow.pytorch.log_model(model, 'model')

2025/10/10 11:55:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/10 11:55:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Door bovenstaande code uit te voeren wordt een map aangemaakt waar de nodige logs in bewaard worden.
Deze kunnen daarna gevisualiseerd worden met behulp van het volgende terminal-commando.
In de output van het commando krijg je een url te zien met de link waar je de tensorboard applicatie kan bekijken.
Met de standaardconfiguratie van de docker container moet je de applicatie kunnen bereiken via de link [localhost:6006](http://localhost:6006)

In [ ]:
!mlflow ui --host 0.0.0.0 --port 6006

/opt/conda/lib/python3.10/site-packages/pydantic/_internal/_config.py:323: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
INFO:     Uvicorn running on http://0.0.0.0:6006 (Press CTRL+C to quit)
INFO:     Started parent process [1124]
INFO:     Started server process [1129]
INFO:     Waiting for application startup.
INFO:     Started server process [1127]
INFO:     Waiting for application startup.
INFO:     Started server process [1128]
INFO:     Application startup complete.
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Application startup complete.
INFO:     Started server process [1126]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     172.19.0.1:49444 - "GET /